In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

import utils
import vision_transformer as vits
from vision_transformer import Attention, DINOHead
from fibottention import get_mask_attn_wythoff

import matplotlib
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

device: mps


In [8]:
EPOCHS = 50
BATCH_SIZE = 32
LR = 5e-6
GLOBAL_SIZE, LOCAL_SIZE = 152, 48
N_LOCAL = 4
OUT_DIM = 2048
TEACHER_MOMENTUM = 0.996
TEACHER_TEMP, STUDENT_TEMP = 0.04, 0.1

In [9]:
normalize = transforms.Compose([
    transforms.ToTensor(),
])

color = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
])

class MultiCrop():
    def __init__(self):
        self.global_t = transforms.Compose([
            transforms.RandomResizedCrop(GLOBAL_SIZE, scale = (0.4, 1.0)), color, 
            utils.GaussianBlur(0.5), normalize])
        
        self.local_t = transforms.Compose([
            transforms.RandomResizedCrop(LOCAL_SIZE, scale = (0.05, 0.4)), color, 
            utils.GaussianBlur(0.5), normalize])
        
    def call(self, img):
        img = img.convert("RGB")
        return [self.global_t(img), self.global_t(img)] + [self.local_t(img) for _ in range(N_LOCAL)]
    
train_set = datasets.ImageFolder("data/train", transforms = MultiCrop())
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, 
                                        num_workers = 0, drop_last = True)

print(len(train_set), "train images,", len(train_set.classes), "classes (unused for training)")


TypeError: ImageFolder.__init__() got an unexpected keyword argument 'transforms'

In [5]:
class FibAttention(Attention):
    def __init__(self, dim, num_heads, depth_id, is_modified=True, is_shuffled=True, **kwargs):
        super().__init__(dim, num_heads=num_heads, **kwargs)
        self.depth_id = depth_id
        self.is_modified = is_modified
        self.is_shuffled = is_shuffled
        self.masks = {}

    def get_mask(self, n_tokens, device):
        key = (n_tokens, str(device))
        if key not in self.masks:
            self.masks[key] = get_mask_attn_wythoff(
                self.num_heads, n_tokens - 1, self.is_modified, self.is_shuffled,
                self.depth_id, add_class_token=True, device=device)
        return self.masks[key]

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        mask = self.get_mask(N, x.device)                      # (heads, N, N), broadcast over batch
        attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj_drop(self.proj(x))
        return x, attn

NameError: name 'GLOBAL_SIZE' is not defined

In [ ]:
def vit_tiny_fib(patch_size=16, **kwargs):
    model = vits.vit_tiny(patch_size=patch_size, **kwargs)
    for i, blk in enumerate(model.blocks):
        blk.attn = FibAttention(model.embed_dim, blk.attn.num_heads, depth_id=i, qkv_bias=True)
    model.apply(model._init_weights)
    return model

# peek at what each head can see in the first layer (global crop: 6x6 patches + CLS)
m = get_mask_attn_wythoff(3, (GLOBAL_SIZE // 16) ** 2, True, True, 0)
print("mask shape:", tuple(m.shape), " fraction of attention kept per head:", m.mean(dim=(1, 2)).tolist())

In [ ]:
def build():
    backbone = vit_tiny_fib(patch_size=16, img_size=[GLOBAL_SIZE])
    return utils.MultiCropWrapper(backbone, DINOHead(backbone.embed_dim, OUT_DIM))

student = build().to(device)
teacher = build().to(device)
teacher.load_state_dict(student.state_dict())  # start teacher = student
for p in teacher.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.04)

In [ ]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, center_momentum=0.9):
        super().__init__()
        self.center_momentum = center_momentum
        self.register_buffer("center", torch.zeros(1, out_dim))

    def forward(self, student_out, teacher_out):
        s = (student_out / STUDENT_TEMP).chunk(2 + N_LOCAL)
        t = F.softmax((teacher_out - self.center) / TEACHER_TEMP, dim=-1).detach().chunk(2)
        loss, n = 0, 0
        for i, q in enumerate(t):
            for v, p in enumerate(s):
                if v == i:
                    continue  # skip same view
                loss += torch.sum(-q * F.log_softmax(p, dim=-1), dim=-1).mean()
                n += 1
        self.update_center(teacher_out)
        return loss / n

    @torch.no_grad()
    def update_center(self, teacher_out):
        batch_center = teacher_out.mean(dim=0, keepdim=True)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

dino_loss = DINOLoss(OUT_DIM).to(device)

In [ ]:
history = []
for epoch in range(EPOCHS):
    student.train()
    total = 0.0
    for crops, _ in train_loader:
        crops = [c.to(device) for c in crops]
        teacher_out = teacher(crops[:2])   # teacher only sees the 2 global crops
        student_out = student(crops)       # student sees all crops
        loss = dino_loss(student_out, teacher_out)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # EMA update of the teacher
        with torch.no_grad():
            for ps, pt in zip(student.parameters(), teacher.parameters()):
                pt.mul_(TEACHER_MOMENTUM).add_(ps.detach(), alpha=1 - TEACHER_MOMENTUM)
        total += loss.item()

    history.append(total / len(train_loader))
    print(f"epoch {epoch + 1}/{EPOCHS}  loss {history[-1]:.4f}")

In [ ]:
torch.save(teacher.backbone.state_dict(), "Models/fib_dino_vit_tiny_teacher.pth")